# 03. QAT STE LSQ LSQ+ PACT and DoReFa

- **Category:** 14_quantisation / 04_vector_lattice_qat_and_lowbit
- **Lineage & Citations:** Bengio et al. (2013, *Estimating or Propagating Gradients Through Stochastic Neurons for Conditional Computation*); Zhou et al. (2016, *DoReFa-Net: Training Low Bitwidth Neural Networks with Low Bitwidth Gradients*); Choi et al. (2018, *PACT: Parameterized Activation Clipping for Quantized Neural Networks*); Esser et al. (2019, *LEARNED STEP SIZE QUANTIZATION*); Bhalgat et al. (2020, *LSQ+: Improving Low-Bit Quantization Through Learnable Offsets and Step Sizes*).
- **Purpose:** Overcome the zero-derivative bottleneck of discrete rounding during training using clamped Straight-Through Estimators (STE), learnable activation clipping (PACT), and analytical gradient-scaled step size quantization (LSQ / LSQ+).
- **Topic Connections:** `🔗 Upstream Prerequisites:` [02. Trellis and E8 Lattice Quantization QuIP# QTIP](02_trellis_and_e8_lattice_quantization_quip_sharp_qtip.ipynb) $\to$ `🔗 Downstream Syntheses:` [04. Binary Sub1bit and Ternary BNN XNOR BiLLM OneBit TWN TTQ](04_binary_sub1bit_and_ternary_bnn_xnor_billm_onebit_twn_ttq.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT use unclamped STE (passing gradients to weights beyond $[q_{\min}s, q_{\max}s]$ causes runaway latent weight growth without changing quantized output); do NOT omit analytical gradient scaling $g = 1/\sqrt{N \cdot Q_P}$ on LSQ step size $s$ (causes wild step-size destabilization and divergent training).

## Mathematical Foundations & Analytical Gradient Scaling

1. **Clamped Straight-Through Estimator (STE)**:
   $$\frac{\partial \widehat{\mathbf{W}}}{\partial \mathbf{W}} = \mathbb{I}[q_{\min} \cdot s \le \mathbf{W} \le q_{\max} \cdot s]$$
2. **Parameterized Activation Clipping (PACT)**:
   $$y = \text{clamp}(x, 0, \alpha), \quad \frac{\partial \widehat{y}}{\partial \alpha} = \mathbb{I}[x \ge \alpha]$$
3. **Learned Step Size Quantization (LSQ)**:
   $$\frac{\partial \bar{v}}{\partial s} = \begin{cases} -Q_N & v/s < -Q_N \\ Q_P & v/s > Q_P \\ \text{round}(v/s) - v/s & \text{otherwise} \end{cases}, \quad g = \frac{1}{\sqrt{N \cdot Q_P}}$$
4. **LSQ+ Learnable Offset Extension**:
   $$\bar{v} = \text{clamp}\left(\text{round}\left(\frac{v - \beta}{s}\right), q_{\min}, q_{\max}\right) \cdot s + \beta$$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

torch.manual_seed(42)
r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
WEIGHTS_PATH = r / "weights/14_quantisation/04_vector_lattice_qat_and_lowbit/03_qat_ste_lsq_lsq_plus_pact_and_dorefa.safetensors"
weights = load_file(str(WEIGHTS_PATH))
print(f"Loaded {len(weights)} tensors from {WEIGHTS_PATH.name}")

In [ ]:
def ste_quantize(w: torch.Tensor, scale: torch.Tensor, qmin: float = -8.0, qmax: float = 7.0):
    q = torch.clamp(torch.round(w / scale), qmin, qmax)
    return q * scale, ((w >= qmin * scale) & (w <= qmax * scale)).float()

def pact_activation(x: torch.Tensor, alpha: torch.Tensor, nbits: int = 4):
    qmax = float(2 ** nbits - 1)
    step = alpha / qmax
    return torch.round(torch.clamp(x, 0.0, float(alpha)) / step) * step, (x >= alpha).float()

def lsq_quantize(w: torch.Tensor, scale: torch.Tensor, qmin: float = -8.0, qmax: float = 7.0):
    v = w / scale
    w_deq = torch.clamp(torch.round(v), qmin, qmax) * scale
    d_s = torch.where(v < qmin, qmin, torch.where(v > qmax, qmax, torch.round(v) - v))
    return w_deq, d_s * (1.0 / (float(w.numel() * qmax) ** 0.5))

def lsq_plus_quantize(w: torch.Tensor, scale: torch.Tensor, beta: torch.Tensor, qmin: float = -8.0, qmax: float = 7.0):
    return torch.clamp(torch.round((w - beta) / scale), qmin, qmax) * scale + beta

def qat_forward(x: torch.Tensor, weight: torch.Tensor, scale_w: torch.Tensor, alpha_x: torch.Tensor, bias: torch.Tensor | None = None):
    x_q, _ = pact_activation(x, alpha_x, nbits=4)
    w_q, _ = lsq_quantize(weight, scale_w, qmin=-8.0, qmax=7.0)
    y = x_q @ w_q
    return ((y + bias) if bias is not None else y), x_q, w_q

In [ ]:
# 1. Linear anchor parity
y = weights["test_input"] @ weights["weight"] + weights["bias"]
assert float(torch.max(torch.abs(y - weights["test_output"]))) < 1e-5

# 2. STE and PACT parity
w_ste, mask_ste = ste_quantize(weights["w_orig"], weights["scale_w"])
assert float(torch.max(torch.abs(w_ste - weights["w_ste_deq"]))) < 1e-5
assert float(torch.max(torch.abs(mask_ste - weights["ste_mask"]))) < 1e-5
x_pact, mask_pact = pact_activation(weights["x_calib"], weights["alpha_x"])
assert float(torch.max(torch.abs(x_pact - weights["x_pact_deq"]))) < 1e-5
assert float(torch.max(torch.abs(mask_pact - weights["pact_grad_mask"]))) < 1e-5

# 3. LSQ and LSQ+ parity
w_lsq, grad_lsq = lsq_quantize(weights["w_orig"], weights["scale_w"])
assert float(torch.max(torch.abs(w_lsq - weights["w_lsq_deq"]))) < 1e-5
assert float(torch.max(torch.abs(grad_lsq - weights["lsq_grad_scale"]))) < 1e-5
w_plus = lsq_plus_quantize(weights["w_orig"], weights["scale_w"], weights["beta_w"])
assert float(torch.max(torch.abs(w_plus - weights["w_lsq_plus"]))) < 1e-5

# 4. QAT forward pass parity
out_qat, _, _ = qat_forward(weights["x_calib"], weights["w_orig"], weights["scale_w"], weights["alpha_x"], weights["b_orig"])
assert float(torch.max(torch.abs(out_qat - weights["out_qat"]))) < 1e-5
print("All STE, PACT, LSQ, and LSQ+ parity assertions verified against golden checkpoint!")

In [ ]:
# Comparative SQNR Benchmark: End-to-End QAT Forward vs FP32 Ground Truth
y_fp32 = weights["out_fp32"]
sig_pwr = float(torch.mean(y_fp32 ** 2))

def eval_sqnr(y_pred: torch.Tensor) -> float:
    mse = float(torch.mean((y_fp32 - y_pred) ** 2))
    return 10.0 * torch.log10(torch.tensor(sig_pwr / (mse + 1e-12))).item()

sqnr_qat = eval_sqnr(weights["out_qat"])
print(f"QAT (PACT Act + LSQ Weight) Forward SQNR: {sqnr_qat:.2f} dB")
assert sqnr_qat > 8.0, "QAT forward pass must achieve >8 dB SQNR at 4-bit precision!"

In [ ]:
# Latency Microbenchmark: Fused PACT + LSQ Forward Pass
b, d_in, d_out = 16, 1024, 1024
x_mock = torch.randn(b, d_in).abs()
w_mock = torch.randn(d_in, d_out) * 0.1
scale_mock = torch.tensor([[0.05]])
alpha_mock = torch.tensor(3.0)
bias_mock = torch.randn(d_out) * 0.01

t0 = time.perf_counter()
for _ in range(50):
    y_mock, _, _ = qat_forward(x_mock, w_mock, scale_mock, alpha_mock, bias_mock)
t_qat = (time.perf_counter() - t0) / 50 * 1e3

print(f"Fused QAT Forward Projection (16x1024x1024) latency: {t_qat:.3f} ms")
assert t_qat < 20.0, "QAT forward projection must complete in <20 ms on CPU"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Gradient Surrogates**: The Straight-Through Estimator enables backpropagation through discrete step functions; clamping prevents latent weight inertia.
2. **Dynamic Range Learning**: PACT continuously optimizes activation clipping thresholds $\alpha$, automatically balancing saturation clipping vs grid precision.
3. **Analytical Scale Normalization**: LSQ parameterizes step size $s$ and stabilizes training via exact $1/\sqrt{N \cdot Q_P}$ gradient scale factors.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [03_qat_ste_lsq_lsq_plus_pact_and_dorefa.md](../../../markdown/14_quantisation/04_vector_lattice_qat_and_lowbit/03_qat_ste_lsq_lsq_plus_pact_and_dorefa.md)
- **PyTorch 2.x Module:** [03_qat_ste_lsq_lsq_plus_pact_and_dorefa.py](../../../pytorch2/14_quantisation/04_vector_lattice_qat_and_lowbit/03_qat_ste_lsq_lsq_plus_pact_and_dorefa.py)
- **Functional JAX Module:** [03_qat_ste_lsq_lsq_plus_pact_and_dorefa.py](../../../jax/14_quantisation/04_vector_lattice_qat_and_lowbit/03_qat_ste_lsq_lsq_plus_pact_and_dorefa.py)
- **Native Rust Candle Track:** [03_qat_ste_lsq_lsq_plus_pact_and_dorefa.rs](../../../candle/14_quantisation/04_vector_lattice_qat_and_lowbit/03_qat_ste_lsq_lsq_plus_pact_and_dorefa.rs)
- **Hardware Kernel (CPU / WGSL):** [03_qat_ste_lsq_lsq_plus_pact_and_dorefa.rs](../../../kernels/14_quantisation/04_vector_lattice_qat_and_lowbit/03_qat_ste_lsq_lsq_plus_pact_and_dorefa.rs)
- **Golden SafeTensors:** [03_qat_ste_lsq_lsq_plus_pact_and_dorefa.safetensors](../../../weights/14_quantisation/04_vector_lattice_qat_and_lowbit/03_qat_ste_lsq_lsq_plus_pact_and_dorefa.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 02. Trellis and E8 Lattice Quantization QuIP# QTIP](02_trellis_and_e8_lattice_quantization_quip_sharp_qtip.ipynb) | [Master Index](../../../README.md#14_quantisation) | [Next: 04. Binary Sub1bit and Ternary BNN XNOR BiLLM OneBit TWN TTQ](04_binary_sub1bit_and_ternary_bnn_xnor_billm_onebit_twn_ttq.ipynb) &rarr;